In [1]:
import os
import pandas as pd
import cv2

In [3]:
images_path = "../RodoSol-ALPR/images"
infos_path = "../RodoSol-ALPR/split.txt"

In [4]:
df = pd.read_csv(infos_path, sep=';', header=None)
print(df.head())

                                 0         1
0  ./images/cars-br/img_000003.jpg  training
1  ./images/cars-br/img_000005.jpg  training
2  ./images/cars-br/img_000007.jpg  training
3  ./images/cars-br/img_000008.jpg  training
4  ./images/cars-br/img_000009.jpg  training


In [5]:
df_train = []
df_valid = []
df_test = []

for i, row in enumerate(df.iterrows()):
    if row[1][1] == "training":
        seps = row[1][0].split("/")
        if "cars" in seps[2]:
            image = seps[3].split(".")[0]
            text_path = os.path.join(images_path, seps[2], image+".txt")
            df_plate = pd.read_csv(text_path, sep=':', header=None)
            plate = df_plate.iloc[1][1].strip()
            corners = df_plate.iloc[3][1].split(" ")[1:]
            df_train.append([row[1][0], plate, corners])

    if row[1][1] == "validation":
        seps = row[1][0].split("/")
        if "cars" in seps[2]:
            image = seps[3].split(".")[0]
            text_path = os.path.join(images_path, seps[2], image+".txt")
            df_plate = pd.read_csv(text_path, sep=':', header=None)
            plate = df_plate.iloc[1][1].strip()
            corners = df_plate.iloc[3][1].split(" ")[1:]
            df_valid.append([row[1][0], plate, corners])

    if row[1][1] == "testing":
        seps = row[1][0].split("/")
        if "cars" in seps[2]:
            image = seps[3].split(".")[0]
            text_path = os.path.join(images_path, seps[2], image+".txt")
            df_plate = pd.read_csv(text_path, sep=':', header=None)
            plate = df_plate.iloc[1][1].strip()
            corners = df_plate.iloc[3][1].split(" ")[1:]
            df_test.append([row[1][0], plate, corners]) 

print(df_train[0])  
print(df_valid[0])

['./images/cars-br/img_000003.jpg', 'PJM6113', ['520,416', '641,417', '643,456', '520,454']]
['./images/cars-br/img_000001.jpg', 'ODE2510', ['558,438', '687,439', '687,482', '558,481']]


In [6]:
import pandas as pd
import os
from pathlib import Path

# Definir caminho para a pasta de treino e validacao
train_path = 'train_c/'
valid_path = 'valid_c/'
test_path = 'test_c/'

# Criar pastas de treino e validacao
Path(train_path).mkdir(parents=True, exist_ok=True)
Path(valid_path).mkdir(parents=True, exist_ok=True)
Path(test_path).mkdir(parents=True, exist_ok=True)

df_train = pd.DataFrame(df_train, columns=['image_path', 'plate_txt', 'bbox'])
df_valid = pd.DataFrame(df_valid, columns=['image_path', 'plate_txt', 'bbox'])
df_test = pd.DataFrame(df_test, columns=['image_path', 'plate_txt', 'bbox'])

In [7]:
csv = []
for idx, row in df_train.iterrows():
    img_path = Path(row['image_path'])
    plate_txt = row['plate_txt']
    bbox_list = row['bbox']

    img = cv2.imread("../RodoSol-ALPR/"+str(img_path))
    if img is None:
        print(f"[ERROR] {img}")
        continue

    points = [tuple(map(int, p.split(','))) for p in bbox_list]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    crop = img[y_min:y_max, x_min:x_max]

    new_name = img_path.stem + img_path.suffix  # ex: img_017505_plate.jpg
    train_path_com = os.path.join(train_path, new_name)
    cv2.imwrite(str(train_path_com), crop)

    csv.append({
        'image_path': os.path.join("train"+"/"+new_name),   # ex: "train/img_017505_plate.jpg"
        'plate_txt': plate_txt
    })

out_df = pd.DataFrame(csv, columns=['image_path', 'plate_txt'])
print(out_df)
out_df.to_csv('train_c.csv', index=False)

                image_path plate_txt
0     train/img_000003.jpg   PJM6113
1     train/img_000005.jpg   ODI9001
2     train/img_000007.jpg   OVJ6688
3     train/img_000008.jpg   ODT4111
4     train/img_000009.jpg   QRD1911
...                    ...       ...
3995  train/img_014992.jpg   MQH5C58
3996  train/img_014995.jpg   MPL9E44
3997  train/img_014996.jpg   PPP8B43
3998  train/img_014997.jpg   PPO9C06
3999  train/img_015000.jpg   QRE2C30

[4000 rows x 2 columns]


In [8]:
csv = []
for idx, row in df_valid.iterrows():
    img_path = Path(row['image_path'])
    plate_txt = row['plate_txt']
    bbox_list = row['bbox']

    img = cv2.imread("../RodoSol-ALPR/"+str(img_path))
    if img is None:
        print(f"[ERROR] {img}")
        continue

    points = [tuple(map(int, p.split(','))) for p in bbox_list]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    crop = img[y_min:y_max, x_min:x_max]

    new_name = img_path.stem + img_path.suffix  # ex: img_017505_plate.jpg
    train_path_com = os.path.join(valid_path, new_name)
    cv2.imwrite(str(train_path_com), crop)

    csv.append({
        'image_path': os.path.join("valid"+"/"+new_name),   # ex: "train/img_017505_plate.jpg"
        'plate_txt': plate_txt
    })

out_df = pd.DataFrame(csv, columns=['image_path', 'plate_txt'])
print(out_df)
out_df.to_csv('valid_c.csv', index=False)

                image_path plate_txt
0     valid/img_000001.jpg   ODE2510
1     valid/img_000006.jpg   GJX9280
2     valid/img_000012.jpg   MTP9115
3     valid/img_000021.jpg   EJE5677
4     valid/img_000023.jpg   PPT7651
...                    ...       ...
1995  valid/img_014975.jpg   LMS1C63
1996  valid/img_014987.jpg   HEI3B13
1997  valid/img_014989.jpg   PPU8I75
1998  valid/img_014994.jpg   MRK8C38
1999  valid/img_014998.jpg   PPW9J90

[2000 rows x 2 columns]


In [9]:
csv = []
for idx, row in df_test.iterrows():
    img_path = Path(row['image_path'])
    plate_txt = row['plate_txt']
    bbox_list = row['bbox']

    img = cv2.imread("../RodoSol-ALPR/"+str(img_path))
    if img is None:
        print(f"[ERROR] {img}")
        continue

    points = [tuple(map(int, p.split(','))) for p in bbox_list]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    crop = img[y_min:y_max, x_min:x_max]

    new_name = img_path.stem + img_path.suffix  # ex: img_017505_plate.jpg
    test_path_com = os.path.join(test_path, new_name)
    cv2.imwrite(str(test_path_com), crop)

    csv.append({
        'image_path': os.path.join("test"+"/"+new_name),   # ex: "train/img_017505_plate.jpg"
        'plate_txt': plate_txt
    })

out_df = pd.DataFrame(csv, columns=['image_path', 'plate_txt'])
print(out_df)
out_df.to_csv('test_c.csv', index=False)

               image_path plate_txt
0     test/img_000236.jpg   MQW8039
1     test/img_000002.jpg   PPC5431
2     test/img_000004.jpg   OVK7900
3     test/img_000010.jpg   OYE3384
4     test/img_000020.jpg   LSE1695
...                   ...       ...
3995  test/img_014972.jpg   QRI0E55
3996  test/img_014973.jpg   QOK1C00
3997  test/img_014985.jpg   QRF2A03
3998  test/img_014993.jpg   LML5E98
3999  test/img_014999.jpg   ODP3D20

[4000 rows x 2 columns]
